# Python LLM API Calls in Dataiku

## Learning Objectives

By completing this notebook, you will:
1. Use Dataiku's LLM Python API to make programmatic calls
2. Handle responses, errors, and retries effectively
3. Implement streaming responses for real-time applications
4. Manage conversation history and context windows
5. Optimize API calls for performance and cost

## Prerequisites

- Module 0 completed (LLM Mesh connections)
- Module 1 completed (Prompt engineering)
- Python programming fundamentals
- Understanding of async/await concepts (helpful but not required)

## Estimated Time: 55 minutes

---

## Why Use Python APIs?

**Beyond the GUI**: While Dataiku's visual tools are great for prototyping, production applications need code.

Python APIs enable:
- **Dynamic prompts** - Build prompts programmatically from data
- **Complex workflows** - Chain multiple LLM calls with logic
- **Custom processing** - Pre/post-process inputs and outputs
- **Integration** - Connect LLMs to databases, APIs, business logic
- **Deployment** - Package as API endpoints, scheduled jobs, or plugins

### Key Insight

**The LLM is just one component** in your application. Python gives you control over the entire pipeline.

### Dataiku LLM Python API

The `dataiku.llm` module provides:
1. **LLM class** - Connect to any LLM Mesh provider
2. **Completion API** - Single-turn text generation
3. **Chat API** - Multi-turn conversations
4. **Streaming API** - Real-time response generation
5. **Usage tracking** - Token counts, costs, latency

## Setup

Import the Dataiku LLM module and helper libraries.

In [ ]:
# Purpose: Import Dataiku LLM API and utilities
# Key Concept: dataiku.llm provides programmatic access to LLM Mesh

import dataiku
from dataiku.llm import LLM, LLMResponse, ConversationHistory
import json
import time
from typing import Dict, List, Generator, Optional
from dataclasses import dataclass
import asyncio

print("✓ Dataiku LLM API loaded")

## Basic LLM Connection

First, establish a connection to your LLM through Dataiku's LLM Mesh.

In [ ]:
# Purpose: Connect to LLM via LLM Mesh
# Key Concept: LLM() accepts connection IDs defined in LLM Mesh settings

# Connect to your LLM Mesh connection
# Replace 'claude-sonnet' with your actual connection ID
llm = LLM('claude-sonnet')

# Basic completion call
response = llm.complete(
    "Explain what an API is in one sentence.",
    max_tokens=100,
    temperature=0.7
)

print("Response:")
print(response.text)
print(f"\nTokens used: {response.usage.total_tokens}")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")

## Understanding LLMResponse Objects

Every LLM call returns an `LLMResponse` object with rich metadata.

In [ ]:
# Purpose: Explore LLMResponse structure
# Key Concept: Responses include text, usage stats, and metadata

response = llm.complete(
    "List three benefits of cloud computing.",
    max_tokens=150,
    temperature=0.5
)

print("=== LLMResponse Attributes ===")
print(f"\ntext: {response.text}")
print(f"\nmodel: {response.model}")
print(f"\nfinish_reason: {response.finish_reason}")
print(f"  (normal, max_tokens, stop_sequence, content_filter)")

print(f"\nusage:")
print(f"  prompt_tokens: {response.usage.prompt_tokens}")
print(f"  completion_tokens: {response.usage.completion_tokens}")
print(f"  total_tokens: {response.usage.total_tokens}")

print(f"\nlatency_ms: {response.latency_ms:.1f}ms")

# Check for successful completion
if response.finish_reason == 'normal':
    print("\n✓ Response completed normally")
elif response.finish_reason == 'max_tokens':
    print("\n⚠ Response was truncated (hit max_tokens limit)")
    print("Consider increasing max_tokens")

## Exercise 1: Dynamic Prompt Generation

**Task**: Create a function that generates commodity analysis prompts from structured data.

Your function should:
- Accept commodity name, price, and change percentage
- Build a well-formatted prompt
- Make the LLM call
- Return parsed JSON response

In [ ]:
# YOUR CODE HERE

def analyze_commodity_movement(commodity: str, current_price: float, 
                               change_pct: float, llm: LLM) -> Dict:
    """
    Generate sentiment analysis for a commodity price movement.
    
    Parameters
    ----------
    commodity : str
        Name of the commodity (e.g., "WTI Crude Oil")
    current_price : float
        Current price
    change_pct : float
        Percentage change (e.g., 3.5 for +3.5%)
    llm : LLM
        Dataiku LLM instance
    
    Returns
    -------
    dict
        Parsed JSON response with sentiment and analysis
    """
    # Step 1: Build the prompt dynamically
    # YOUR CODE HERE
    prompt = f"""You are a commodity market analyst.

Analyze this price movement:
Commodity: {commodity}
Current Price: ${current_price}
Change: {change_pct:+.1f}%

Provide analysis in JSON format:
{{
  "sentiment": "bullish" | "bearish" | "neutral",
  "magnitude": "small" | "moderate" | "large",
  "likely_drivers": ["list", "of", "factors"],
  "short_summary": "one sentence"
}}

JSON:"""
    
    # Step 2: Make the LLM call
    response = llm.complete(prompt, max_tokens=250, temperature=0.3)
    
    # Step 3: Parse the response
    try:
        result = json.loads(response.text)
        result['_metadata'] = {
            'tokens_used': response.usage.total_tokens,
            'latency_ms': response.latency_ms
        }
        return result
    except json.JSONDecodeError:
        return {
            'error': 'Failed to parse JSON response',
            'raw_response': response.text
        }

# Test cases
test_scenarios = [
    {'commodity': 'WTI Crude Oil', 'current_price': 82.45, 'change_pct': 5.2},
    {'commodity': 'Natural Gas', 'current_price': 2.34, 'change_pct': -8.1},
    {'commodity': 'Gold', 'current_price': 2050, 'change_pct': 0.3}
]

print("Testing dynamic prompt generation...\n")
for scenario in test_scenarios:
    result = analyze_commodity_movement(**scenario, llm=llm)
    
    print(f"{scenario['commodity']} ({scenario['change_pct']:+.1f}%):")
    if 'error' not in result:
        print(f"  Sentiment: {result['sentiment']} ({result['magnitude']})")
        print(f"  Summary: {result['short_summary']}")
        print(f"  Tokens: {result['_metadata']['tokens_used']}")
    else:
        print(f"  Error: {result['error']}")
    print()

# Auto-graded checks
test_result = analyze_commodity_movement('Test Commodity', 100.0, 5.0, llm)
assert 'sentiment' in test_result, "Response should include sentiment field"
assert 'error' not in test_result, "Function should successfully parse response"

print("✓ Exercise 1 passed!")

## Error Handling and Retries

Production code must handle failures gracefully:
- API timeouts
- Rate limits
- Invalid responses
- Network errors

Implement retry logic with exponential backoff.

In [ ]:
# Purpose: Robust error handling for LLM calls
# Key Concept: Production code needs retry logic and graceful degradation

class LLMCallError(Exception):
    """Custom exception for LLM call failures."""
    pass

def call_llm_with_retry(
    llm: LLM,
    prompt: str,
    max_retries: int = 3,
    initial_delay: float = 1.0,
    **llm_kwargs
) -> LLMResponse:
    """
    Make LLM call with exponential backoff retry logic.
    
    Parameters
    ----------
    llm : LLM
        Dataiku LLM instance
    prompt : str
        Prompt to send
    max_retries : int
        Maximum retry attempts
    initial_delay : float
        Initial retry delay in seconds
    **llm_kwargs
        Additional arguments for llm.complete()
    
    Returns
    -------
    LLMResponse
        Successful response
    
    Raises
    ------
    LLMCallError
        If all retries fail
    """
    delay = initial_delay
    last_error = None
    
    for attempt in range(max_retries):
        try:
            response = llm.complete(prompt, **llm_kwargs)
            
            # Validate response
            if response.finish_reason == 'content_filter':
                raise LLMCallError("Content filtered by provider")
            
            return response
            
        except Exception as e:
            last_error = e
            
            if attempt < max_retries - 1:
                print(f"Attempt {attempt + 1} failed: {e}")
                print(f"Retrying in {delay:.1f}s...")
                time.sleep(delay)
                delay *= 2  # Exponential backoff
            else:
                print(f"All {max_retries} attempts failed")
    
    raise LLMCallError(f"Failed after {max_retries} attempts: {last_error}")

# Test with a valid prompt
try:
    response = call_llm_with_retry(
        llm,
        "Say 'Hello from retry logic!' in a professional tone.",
        max_tokens=50,
        temperature=0.7
    )
    print("Success:")
    print(response.text)
except LLMCallError as e:
    print(f"Failed: {e}")

print("\n✓ Retry logic implemented")

## Exercise 2: Batch Processing with Error Handling

**Task**: Process a batch of commodity reports with robust error handling.

Your function should:
- Process multiple reports
- Handle individual failures without stopping
- Track success/failure rates
- Return both results and errors

In [ ]:
# YOUR CODE HERE

@dataclass
class BatchResult:
    """Results from batch processing."""
    successful: List[Dict]
    failed: List[Dict]
    total_processed: int
    success_rate: float
    total_tokens: int
    total_time_ms: float

def process_reports_batch(reports: List[str], llm: LLM) -> BatchResult:
    """
    Process multiple reports with error handling.
    
    Parameters
    ----------
    reports : List[str]
        List of report texts to analyze
    llm : LLM
        Dataiku LLM instance
    
    Returns
    -------
    BatchResult
        Aggregated results with success/failure tracking
    """
    successful = []
    failed = []
    total_tokens = 0
    start_time = time.time()
    
    # YOUR CODE HERE
    # Process each report with error handling
    for i, report in enumerate(reports):
        try:
            prompt = f"""Extract key information from this commodity report.

Report: {report}

Respond in JSON:
{{
  "commodity": "name",
  "sentiment": "bullish/bearish/neutral",
  "key_fact": "most important information"
}}

JSON:"""
            
            response = call_llm_with_retry(
                llm, prompt, max_tokens=200, temperature=0.2, max_retries=2
            )
            
            parsed = json.loads(response.text)
            parsed['_id'] = i
            parsed['_tokens'] = response.usage.total_tokens
            
            successful.append(parsed)
            total_tokens += response.usage.total_tokens
            
        except Exception as e:
            failed.append({
                '_id': i,
                'report': report[:100],
                'error': str(e)
            })
    
    total_time_ms = (time.time() - start_time) * 1000
    total_processed = len(reports)
    success_rate = len(successful) / total_processed if total_processed > 0 else 0
    
    return BatchResult(
        successful=successful,
        failed=failed,
        total_processed=total_processed,
        success_rate=success_rate,
        total_tokens=total_tokens,
        total_time_ms=total_time_ms
    )

# Test with sample reports
sample_reports = [
    "Crude oil futures rose 3.2% on supply concerns from the Middle East.",
    "Natural gas prices fell sharply as mild weather reduced heating demand.",
    "Gold hit a new record high above $2,100 per ounce on safe-haven buying.",
    "Copper inventories declined for the third consecutive week.",
    "This is an invalid report with no useful information xyz123"  # Edge case
]

print("Processing batch...\n")
results = process_reports_batch(sample_reports, llm)

print("=== Batch Processing Results ===")
print(f"Total processed: {results.total_processed}")
print(f"Successful: {len(results.successful)}")
print(f"Failed: {len(results.failed)}")
print(f"Success rate: {results.success_rate:.1%}")
print(f"Total tokens: {results.total_tokens}")
print(f"Total time: {results.total_time_ms:.0f}ms")
print(f"Avg time per report: {results.total_time_ms / results.total_processed:.0f}ms")

if results.successful:
    print("\nSample successful result:")
    print(json.dumps(results.successful[0], indent=2))

if results.failed:
    print("\nFailed reports:")
    for failure in results.failed:
        print(f"  ID {failure['_id']}: {failure['error']}")

# Auto-graded checks
assert results.total_processed == len(sample_reports), "Should process all reports"
assert len(results.successful) + len(results.failed) == results.total_processed, "All reports should be categorized"
assert results.success_rate > 0, "Should have some successful results"

print("\n✓ Exercise 2 passed!")

## Streaming Responses

For real-time applications, stream responses as they're generated instead of waiting for completion.

In [ ]:
# Purpose: Demonstrate streaming API
# Key Concept: Streaming provides better UX for long responses

def stream_llm_response(llm: LLM, prompt: str, **kwargs) -> Generator[str, None, None]:
    """
    Stream LLM response token by token.
    
    Yields
    ------
    str
        Text chunks as they arrive
    """
    # Dataiku streaming API (simulated)
    # In actual Dataiku, use: llm.complete_stream(prompt, **kwargs)
    
    # For this demo, we'll simulate streaming by chunking the response
    response = llm.complete(prompt, **kwargs)
    
    # Simulate streaming by yielding words
    words = response.text.split()
    for i, word in enumerate(words):
        yield word + (" " if i < len(words) - 1 else "")
        time.sleep(0.05)  # Simulate network delay

# Example usage
prompt = """Explain the concept of 'contango' in commodity futures markets 
in 2-3 sentences."""

print("Streaming response:")
print("-" * 60)

full_response = ""
for chunk in stream_llm_response(llm, prompt, max_tokens=200, temperature=0.7):
    print(chunk, end='', flush=True)
    full_response += chunk

print("\n" + "-" * 60)
print("\n✓ Streaming complete")

## Managing Conversation History

For multi-turn conversations, maintain context across multiple calls.

In [ ]:
# Purpose: Multi-turn conversation management
# Key Concept: Conversation history enables context-aware interactions

class ConversationManager:
    """
    Manage multi-turn conversations with context window limits.
    """
    
    def __init__(self, llm: LLM, system_prompt: str = "", max_history: int = 10):
        self.llm = llm
        self.system_prompt = system_prompt
        self.max_history = max_history
        self.messages: List[Dict[str, str]] = []
        
        if system_prompt:
            self.messages.append({'role': 'system', 'content': system_prompt})
    
    def send_message(self, user_message: str, **llm_kwargs) -> str:
        """
        Send a message and get response.
        
        Automatically manages conversation history.
        """
        # Add user message
        self.messages.append({'role': 'user', 'content': user_message})
        
        # Build prompt from conversation history
        prompt = self._build_prompt()
        
        # Get response
        response = self.llm.complete(prompt, **llm_kwargs)
        
        # Add assistant response to history
        self.messages.append({'role': 'assistant', 'content': response.text})
        
        # Trim history if too long
        self._trim_history()
        
        return response.text
    
    def _build_prompt(self) -> str:
        """Build prompt from message history."""
        prompt_parts = []
        
        for msg in self.messages:
            if msg['role'] == 'system':
                prompt_parts.append(f"System: {msg['content']}")
            elif msg['role'] == 'user':
                prompt_parts.append(f"\nUser: {msg['content']}")
            elif msg['role'] == 'assistant':
                prompt_parts.append(f"\nAssistant: {msg['content']}")
        
        prompt_parts.append("\nAssistant:")
        return "\n".join(prompt_parts)
    
    def _trim_history(self):
        """Keep only recent messages within max_history limit."""
        # Always keep system message
        system_msgs = [m for m in self.messages if m['role'] == 'system']
        other_msgs = [m for m in self.messages if m['role'] != 'system']
        
        if len(other_msgs) > self.max_history:
            other_msgs = other_msgs[-self.max_history:]
        
        self.messages = system_msgs + other_msgs
    
    def clear_history(self):
        """Clear conversation history (keep system prompt)."""
        self.messages = [m for m in self.messages if m['role'] == 'system']

# Example: Commodity Q&A conversation
conversation = ConversationManager(
    llm,
    system_prompt="You are a commodity market expert. Provide concise, accurate answers.",
    max_history=6
)

print("=== Conversation Demo ===")
print()

questions = [
    "What factors influence crude oil prices?",
    "Which of those factors is most important right now?",
    "How do OPEC decisions impact that factor?"
]

for i, question in enumerate(questions, 1):
    print(f"Q{i}: {question}")
    answer = conversation.send_message(question, max_tokens=150, temperature=0.7)
    print(f"A{i}: {answer}")
    print()

print(f"Conversation history length: {len(conversation.messages)} messages")
print("\n✓ Conversation management demonstrated")

## Exercise 3: Build a Commodity Analysis Bot

**Task**: Create an interactive bot that answers questions about commodity markets.

Your bot should:
- Maintain conversation context
- Handle follow-up questions
- Provide structured responses
- Track token usage

In [ ]:
# YOUR CODE HERE

class CommodityBot:
    """
    Interactive bot for commodity market analysis.
    """
    
    def __init__(self, llm: LLM):
        # YOUR CODE HERE
        self.conversation = ConversationManager(
            llm,
            system_prompt="""You are an expert commodity market analyst.
            
Provide concise, data-driven insights about:
- Price movements and trends
- Supply and demand factors
- Market fundamentals
- Trading strategies

Always be specific and cite reasoning.""",
            max_history=8
        )
        self.total_tokens = 0
        self.query_count = 0
    
    def ask(self, question: str) -> Dict:
        """
        Ask the bot a question.
        
        Returns
        -------
        dict
            answer, tokens_used, conversation_length
        """
        # YOUR CODE HERE
        # Get response with token tracking
        response = self.conversation.send_message(
            question,
            max_tokens=300,
            temperature=0.6
        )
        
        self.query_count += 1
        # Note: In actual Dataiku, we'd get token count from response
        estimated_tokens = len(question.split()) + len(response.split()) * 2
        self.total_tokens += estimated_tokens
        
        return {
            'answer': response,
            'tokens_used': estimated_tokens,
            'conversation_length': len(self.conversation.messages),
            'query_number': self.query_count
        }
    
    def reset(self):
        """Reset conversation history."""
        self.conversation.clear_history()
        self.query_count = 0
    
    def get_stats(self) -> Dict:
        """Get usage statistics."""
        return {
            'total_queries': self.query_count,
            'total_tokens': self.total_tokens,
            'avg_tokens_per_query': self.total_tokens / self.query_count if self.query_count > 0 else 0,
            'conversation_length': len(self.conversation.messages)
        }

# Test the bot
bot = CommodityBot(llm)

print("=== Commodity Bot Demo ===")
print()

test_questions = [
    "What's the relationship between oil prices and the dollar?",
    "Why does that relationship exist?",
    "Give me a specific example from recent history."
]

for question in test_questions:
    result = bot.ask(question)
    print(f"Q{result['query_number']}: {question}")
    print(f"A: {result['answer'][:200]}..." if len(result['answer']) > 200 else f"A: {result['answer']}")
    print(f"   (Tokens: {result['tokens_used']}, Conversation length: {result['conversation_length']})")
    print()

# Display statistics
stats = bot.get_stats()
print("=== Bot Statistics ===")
for key, value in stats.items():
    print(f"{key}: {value}")

# Auto-graded checks
assert bot.query_count == len(test_questions), "Should track query count"
assert bot.total_tokens > 0, "Should track token usage"
assert len(bot.conversation.messages) > len(test_questions), "Should maintain conversation history"

print("\n✓ Exercise 3 passed!")

## Summary

Congratulations! You've mastered Dataiku's LLM Python API. Key takeaways:

1. **LLM class** - Connect to any LLM Mesh provider programmatically
2. **LLMResponse** - Rich metadata including tokens, latency, finish reason
3. **Error handling** - Retry logic and graceful degradation are essential
4. **Batch processing** - Process multiple requests with failure isolation
5. **Streaming** - Better UX for long responses
6. **Conversation management** - Maintain context across multiple turns

## Best Practices

- Always use retry logic with exponential backoff
- Track token usage to monitor costs
- Implement timeouts for production systems
- Validate responses before using them
- Trim conversation history to manage context windows
- Log errors for debugging and monitoring
- Use streaming for responses > 200 tokens

## Common Patterns

**Data Enrichment Pipeline**:
```python
1. Read data from Dataiku dataset
2. For each row, call LLM with retry logic
3. Parse and validate response
4. Write enriched data to output dataset
5. Log failures for review
```

**API Endpoint**:
```python
1. Receive request via Dataiku API endpoint
2. Build prompt from request parameters
3. Call LLM with appropriate settings
4. Format response as JSON
5. Return with proper status codes
```

## Next Steps

- **Notebook 02**: Build a complete custom LLM application
- **Module 4**: Deploy your application as a production API